# **|| AI Multi-Agent Research Assistant  ||**

This project builds a research assistant using **4 AI agents** that work together:

| Agent | What it does |
|-------|------|
|  **Planner Agent** | Breaks the topic into smaller search queries |
|  **Search Agent** | Searches the web using DuckDuckGo |
|  **Summarizer Agent** | Summarizes each result using an LLM |
|  **Report Agent** | Combines everything into a final report |

---
**Tech used:** `groq` , `duckduckgo-search` , `rich`  
**Model:** Llama 3 via Groq (free tier)


## **1** - ***Installation***

In [ ]:
%%capture
!pip install groq ddgs rich

## **2** — ***API Key Setup***

> Free Groq API key at https://console.groq.com

In [ ]:
import os
from getpass import getpass

# Using getpass so the key doesn't show up on screen
GROQ_API_KEY = getpass("Enter your Groq API key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("API key saved!")

## **3** — ***Imports and Settings***

In [ ]:
import os
os.environ["PYTHONIOENCODING"] = "utf-8"

import json
import time
from datetime import datetime

from groq import Groq
from ddgs import DDGS
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.progress import track
from rich.markdown import Markdown

# --- Settings ---
MODEL_NAME = "llama-3.1-8b-instant"   # Free model on Groq
MAX_TOKENS = 1024
TEMPERATURE = 0.4
MAX_SEARCH_RESULTS = 5   # How many results to fetch per query
MAX_QUERIES = 3          # How many sub-queries the planner will create

console = Console()
client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("All imports done!")

## **4** — ***Helper Classes to Store Data***

These are simple classes to keep search results and summaries organized instead of using messy nested dictionaries.

In [ ]:
class SearchResult:
    """Holds one web search result"""
    def __init__(self, title, url, snippet, query):
        self.title = title
        self.url = url
        self.snippet = snippet
        self.query = query  # Which sub-query this came from


class SummarizedSource:
    """Holds a search result after it has been summarized by the LLM"""
    def __init__(self, title, url, summary, query):
        self.title = title
        self.url = url
        self.summary = summary
        self.query = query


class ResearchReport:
    """Holds the final report and all the info that went into it"""
    def __init__(self, topic, queries, sources, report_text):
        self.topic = topic
        self.queries = queries
        self.sources = sources
        self.report_text = report_text
        self.generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


print("Data classes ready!")

## **5** — ***LLM Helper Function***

A simple function to call the Groq API. All 4 agents use this.

In [ ]:
def call_llm(system_prompt, user_prompt):
    """
    Sends a message to the LLM and returns its response as a string.
    Has a basic retry in case the API throws an error.
    """
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE
        )
        return response.choices[0].message.content.strip()

    except Exception as e:
        print(f"API error: {e}. Waiting 5 seconds and trying again...")
        time.sleep(5)
        # Try one more time
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE
        )
        return response.choices[0].message.content.strip()


print("LLM helper ready!")

## **6** — ***The 4 Agents***

Each agent is a class with a `run()` method. They are designed to do one job each and pass their output to the next agent.

In [ ]:
# =============================================================================
# AGENT 1: PLANNER AGENT
# Takes the user's topic and breaks it into smaller, specific search queries
# =============================================================================

class PlannerAgent:

    system_prompt = """
        You are a research planning assistant. Your job is to take a broad topic
        and break it down into specific search queries for a search engine.

        Rules:
        - Return ONLY a JSON array of strings. No explanation, no extra text.
        - Each string should be a short search query (around 5-8 words).
        - Each query should cover a different angle of the topic.
        - Example output: ["query one here", "query two here", "query three here"]
    """

    def run(self, topic, n_queries=MAX_QUERIES):
        console.print(Panel(f"[bold cyan]Planner Agent[/bold cyan] — Breaking topic into {n_queries} queries", expand=False))

        user_prompt = f'Topic: "{topic}"\nGenerate {n_queries} search queries as a JSON array.'
        response = call_llm(self.system_prompt, user_prompt)

        # Try to parse the JSON response
        try:
            # Sometimes the model wraps with ```json ... ``` so we clean that
            clean = response.strip()
            if clean.startswith("```"):
                clean = clean.split("```")[1]
                if clean.startswith("json"):
                    clean = clean[4:]
            queries = json.loads(clean)
            queries = queries[:n_queries]  # Make sure we don't go over the limit
        except Exception as e:
            print(f"Couldn't parse JSON ({e}), splitting by lines instead")
            # Fallback: just split by newlines and clean each line
            queries = [line.strip().strip('",-') for line in response.splitlines() if line.strip()]
            queries = queries[:n_queries]

        # Show the queries in a nice table
        table = Table(title="Research Plan", show_lines=True)
        table.add_column("#", style="dim", width=4)
        table.add_column("Query")
        for i, q in enumerate(queries, 1):
            table.add_row(str(i), q)
        console.print(table)

        return queries


# =============================================================================
# AGENT 2: SEARCH AGENT
# Takes each query and fetches real web results using DuckDuckGo
# =============================================================================

class SearchAgent:

    def run(self, queries, max_results=MAX_SEARCH_RESULTS):
        console.print(Panel("[bold cyan]Search Agent[/bold cyan] — Fetching web results", expand=False))

        all_results = []

        for query in track(queries, description="Searching the web..."):
            try:
                with DDGS() as ddgs:
                    hits = list(ddgs.text(query, max_results=max_results))

                for hit in hits:
                    result = SearchResult(
                        title   = hit.get("title", "No title"),
                        url     = hit.get("href", ""),
                        snippet = hit.get("body", "")[:600],  # Trim long snippets
                        query   = query
                    )
                    all_results.append(result)

                time.sleep(1)  # Small delay to avoid getting blocked

            except Exception as e:
                print(f"Search failed for '{query}': {e}")

        print(f"Found {len(all_results)} results total")
        return all_results


# =============================================================================
# AGENT 3: SUMMARIZER AGENT
# Reads each search result and uses the LLM to write a short summary
# =============================================================================

class SummarizerAgent:

    system_prompt = """
        You are a research analyst. You will be given the title, URL, and a snippet
        from a webpage. Write a short 2-3 sentence summary of what the page is about
        and what useful information it contains.
        Only use the information given to you. Do not make things up.
    """

    def run(self, results):
        console.print(Panel(f"[bold cyan]Summarizer Agent[/bold cyan] — Summarizing {len(results)} sources", expand=False))

        summarized = []

        for result in track(results, description="Summarizing sources..."):
            # Skip results with no content
            if not result.snippet:
                continue

            user_prompt = f"""Search query: {result.query}
Title: {result.title}
URL: {result.url}
Snippet: {result.snippet}"""

            summary = call_llm(self.system_prompt, user_prompt)

            summarized.append(SummarizedSource(
                title   = result.title,
                url     = result.url,
                summary = summary,
                query   = result.query
            ))

            time.sleep(0.3)  # Small pause between API calls

        print(f"Summarized {len(summarized)} sources")
        return summarized


# =============================================================================
# AGENT 4: REPORT AGENT
# Takes all the summaries and writes a full structured research report
# =============================================================================

class ReportAgent:

    system_prompt = """
        You are a research writer. Using the source summaries provided, write a
        well-structured research report in Markdown format.

        The report must have these sections:
        1. ## Executive Summary
        2. ## Key Findings
        3. ## Detailed Analysis
        4. ## Current Trends
        5. ## Conclusion
        6. ## Sources

        Guidelines:
        - Only use information from the summaries given to you.
        - Write clearly and professionally.
        - Cite sources using [1], [2] etc. where relevant.
    """

    def run(self, topic, queries, sources):
        console.print(Panel("[bold cyan]Report Agent[/bold cyan] — Writing final report", expand=False))

        # Build a big string with all the summaries
        sources_text = ""
        for i, src in enumerate(sources, 1):
            sources_text += f"\n[{i}] {src.title}\nURL: {src.url}\nSummary: {src.summary}\n"

        user_prompt = f"""Research Topic: {topic}
Queries used: {', '.join(queries)}

Source Summaries:
{sources_text}"""

        report_text = call_llm(self.system_prompt, user_prompt)

        return ResearchReport(
            topic       = topic,
            queries     = queries,
            sources     = sources,
            report_text = report_text
        )


print("All 4 agents defined!")

## **7** — ***Connecting All Together***

This class connects all 4 agents together. You just give it a topic and it handles the rest.

In [ ]:
class ResearchOrchestrator:
    """
    Runs the full research pipeline:
    Planner -> Search -> Summarizer -> Report
    """

    def __init__(self):
        self.planner    = PlannerAgent()
        self.searcher   = SearchAgent()
        self.summarizer = SummarizerAgent()
        self.reporter   = ReportAgent()

    def run(self, topic):
        start_time = time.time()

        console.print(Panel(
            f"[bold magenta]Starting Research Pipeline[/bold magenta]\nTopic: {topic}",
            expand=False
        ))

        # Step 1: Plan the queries
        queries = self.planner.run(topic)

        # Step 2: Search the web
        raw_results = self.searcher.run(queries)

        if len(raw_results) == 0:
            print("No search results found! Try a different topic.")
            return None

        # Step 3: Summarize each result
        summarized = self.summarizer.run(raw_results)

        # Step 4: Write the final report
        report = self.reporter.run(topic, queries, summarized)

        total_time = round(time.time() - start_time, 1)
        console.print(Panel(
            f"[bold green]Done![/bold green]  Time: {total_time}s  |  Sources used: {len(summarized)}",
            expand=False
        ))

        return report


print("Orchestrator ready!")

## **8** — ***Run It!***

Change the `TOPIC` variable below to whatever you want to research.

In [ ]:
# --- Change this to your research topic ---
TOPIC = "The impact of large language models on software development in 2024"

orchestrator = ResearchOrchestrator()
report = None # Initialize report to None to prevent NameError if orchestration fails

try:
    report = orchestrator.run(TOPIC)
except Exception as e:
    print(f"An error occurred during research orchestration: {e}")
    report = None # Ensure report is still None if an exception occurs

## **9** — ***View the Report***

In [ ]:
if report:
    console.print(Panel(
        f"[bright_cyan]Research Report[/bright_cyan]\n"
        f"Topic: {report.topic}\n"
        f"Generated at: {report.generated_at}",
        expand=False
    ))
    console.print(Markdown(report.report_text))
else:
    print("No report was generated.")

## **10** — ***Save the Report***

Saves the report as a `.md` file and a `.json` file. In Colab it will automatically download both.

In [ ]:
def save_report(report):
    # Create a clean filename from the topic
    topic_short = report.topic[:40].replace(" ", "_")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # --- Save as markdown ---
    md_filename = f"report_{topic_short}_{timestamp}.md"
    with open(md_filename, "w") as f:
        f.write(f"# Research Report: {report.topic}\n")
        f.write(f"Generated: {report.generated_at}\n\n")
        f.write(report.report_text)

    # --- Save as json (useful if you want to process it later) ---
    json_filename = f"report_{topic_short}_{timestamp}.json"
    report_data = {
        "topic": report.topic,
        "generated_at": report.generated_at,
        "queries": report.queries,
        "report_text": report.report_text,
        "sources": [
            {"title": s.title, "url": s.url, "summary": s.summary}
            for s in report.sources
        ]
    }
    with open(json_filename, "w") as f:
        json.dump(report_data, f, indent=2)

    print(f"Saved: {md_filename}")
    print(f"Saved: {json_filename}")
    return md_filename, json_filename


if report:
    md_file, json_file = save_report(report)

    # Try to auto-download if we're in Colab
    try:
        from google.colab import files
        files.download(md_file)
        files.download(json_file)
        print("Download started!")
    except ImportError:
        print("Not in Colab — files saved in current directory.")

## ****Things You Can Change!***

- `TOPIC` in Step 8 — your research question

- `MAX_QUERIES` in Step 3 — more queries = wider research but takes longer

- `MAX_SEARCH_RESULTS` in Step 3 — more results per query

- The `system_prompt` inside each agent class to change how they behave
